# WP49 — Multi-Agent Strange Loop: Theory of Mind

**Prometheus v0 PoC · Work Package 49**

> *"A strange loop occurs whenever … we unexpectedly find ourselves back where we started."*
> — Douglas Hofstadter, *Gödel, Escher, Bach*, 1979, Chapter 19

## What WP49 does

WP29 (Self-Play Tournament) has Alpha and Beta competing via Elo-rated outcomes.  Neither agent models the *other's policy*.

WP49 adds **Theory of Mind (k=2)**:

- **Alpha** maintains a model of **Beta's** action distribution.
- **Beta** maintains a model of **Alpha's** action distribution.
- Each agent **conditions its own action** on its prediction of the opponent.
- Alpha models Beta's model of Alpha — the **second-order Strange Loop** closes at the inter-agent level.

### Adversarial vs. Cooperative Mode

| Situation | Strategy |
|-----------|----------|
| Ahead on Elo | Adversarial: counter the predicted opponent move |
| Behind on Elo | Cooperative: mirror the predicted opponent move |


In [ ]:
import sys, os, pathlib

def _find_repo_root() -> str:
    """Return the Prometheus_v0_PoC repo root regardless of kernel CWD."""
    candidates = [
        # Running from inside notebooks/
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        # Running from repo root
        os.getcwd(),
        # Absolute fallback — known install location
        str(pathlib.Path.home() / "Prometheus_v0_PoC"),
        "/home/pmc/Prometheus_v0_PoC",
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, "prometheus")):
            return c
    raise RuntimeError(
        "Cannot locate Prometheus_v0_PoC repo root. "
        f"Tried: {candidates}"
    )

repo_root = _find_repo_root()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import logging
logging.basicConfig(level=logging.WARNING)

from prometheus.wp49_theory_of_mind import (
    run_tom_tournament,
    verify_wp49_exit_criteria,
    MindReadingAgent,
    TheoryOfMindTournament,
)
print("WP49 loaded ✓")


## Run the Theory-of-Mind Tournament

In [ ]:
report = run_tom_tournament(n_matches=60, seed=42, early_stop=False)
print(report.summary())


## Tournament Dynamics

Elo trajectory + prediction accuracy over time.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use("Agg")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

recs = report.records
matches = [r.match_number   for r in recs]
a_elo   = [r.alpha_elo      for r in recs]
b_elo   = [r.beta_elo       for r in recs]
a_acc   = [r.alpha_accuracy for r in recs]
b_acc   = [r.beta_accuracy  for r in recs]
a_ent   = [r.alpha_model_entropy for r in recs]
b_ent   = [r.beta_model_entropy  for r in recs]

# Rolling prediction accuracy (window=5)
def rolling_pred(recs, key, window=5):
    vals = [getattr(r, key) for r in recs]
    result = []
    for i in range(len(vals)):
        w = vals[max(0, i-window): i+1]
        result.append(sum(w) / len(w))
    return result

a_pred_roll = rolling_pred(recs, "alpha_pred_correct")
b_pred_roll = rolling_pred(recs, "beta_pred_correct")

if HAS_MPL:
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # Panel 1: Elo
    axes[0].plot(matches, a_elo, "b-", label="Alpha Elo", linewidth=1.5)
    axes[0].plot(matches, b_elo, "r-", label="Beta Elo",  linewidth=1.5)
    axes[0].set_ylabel("Elo Rating")
    axes[0].set_title("Theory-of-Mind Tournament — Second-Order Strange Loop")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # Panel 2: Accuracy
    axes[1].plot(matches, a_acc, "b-", alpha=0.7, label="Alpha accuracy")
    axes[1].plot(matches, b_acc, "r-", alpha=0.7, label="Beta accuracy")
    axes[1].set_ylabel("Synthesis Accuracy")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    # Panel 3: Prediction accuracy (rolling)
    axes[2].plot(matches, a_pred_roll, "b-", label="Alpha pred acc (rolling)")
    axes[2].plot(matches, b_pred_roll, "r-", label="Beta pred acc (rolling)")
    axes[2].axhline(1.0 / 5, color="gray", linestyle="--", alpha=0.6, label="Chance (1/5)")
    axes[2].set_ylabel("Prediction Accuracy")
    axes[2].set_xlabel("Match Number")
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig("wp49_tom_tournament.png", dpi=100)
    plt.show()
    print("Plot saved → wp49_tom_tournament.png")
else:
    print("matplotlib not available")
    for r in recs[:10]:
        print(f"Match {r.match_number:3d}: α_acc={r.alpha_accuracy:.3f} β_acc={r.beta_accuracy:.3f} "
              f"α_pred={'✓' if r.alpha_pred_correct else '✗'} β_pred={'✓' if r.beta_pred_correct else '✗'} "
              f"winner={r.winner}")


## Opponent Model Entropy

As each agent learns the other's policy, entropy should *decrease* (more confident predictions).

In [ ]:
import math
H_uniform = math.log(5)  # entropy if policy were uniform over 5 actions
print(f"Uniform entropy (prior)    : {H_uniform:.4f}")
print(f"Mean entropy Alpha model   : {report.mean_model_entropy_alpha:.4f}")
print(f"Mean entropy Beta model    : {report.mean_model_entropy_beta:.4f}")
print()
print(f"Alpha learned: entropy reduced by {(H_uniform - report.mean_model_entropy_alpha) / H_uniform:.1%}")
print(f"Beta learned : entropy reduced by {(H_uniform - report.mean_model_entropy_beta) / H_uniform:.1%}")
print()
print(f"Alpha prediction accuracy : {report.alpha_prediction_acc:.3f}")
print(f"Beta prediction accuracy  : {report.beta_prediction_acc:.3f}")


## Hofstadter Statement — The Second-Order Strange Loop

In [ ]:
print(report.hofstadter_statement)


## WP49 Exit Criteria

In [ ]:
criteria = verify_wp49_exit_criteria(report)
all_pass = all(criteria.values())
print(f"{'PASS' if all_pass else 'FAIL'} — WP49 Exit Criteria")
print()
for name, result in criteria.items():
    status = "✓" if result else "✗"
    print(f"  [{status}] {name}")
print()
print(f"All criteria pass: {all_pass}")


## Conclusion

WP49 implements Hofstadter's **second-order strange loop** at the inter-agent level:

- Each agent maintains an **opponent model** (EMA over observed actions).
- Each agent **conditions its own action** on its model of the opponent.
- The opponent model is itself conditioned on knowing that *the opponent is modelling them*.
- **A models B models A** — the loop closes.

**Camerer et al. (2004)**: k-level reasoning (k=2) improves strategic outcomes over k=0 (no theory of mind).  WP49 demonstrates this computationally: prediction accuracy significantly above chance, and entropy of opponent models well below the uniform prior.
